# Ноутбук для создания и изучения нейронных сетей

PyTorch, CPU-only (нет GPU), но код пишем device-agnostic через torch.device. Для валидации по ходу экспериментов (архитектура, batchnorm, dropout, оптимизаторы, LR) используем один holdout-сплит, чтобы быстро итерироваться. В конце, когда архитектура зафиксирована, прогоним лучший вариант через 5-fold CV, как остальные модели, чтобы можно было сравнить в общей таблице

In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from preprocessing import preprocess_data_advanced

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

Загрузим данные и применим тот же зафиксированный препроцессинг, что и в classic_models_exploration

In [2]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")
test_passenger_ids = test_data['PassengerId']

train_data, artifacts = preprocess_data_advanced(train_data, is_train=True)
test_data = preprocess_data_advanced(test_data, is_train=False, artifacts=artifacts)

categorical_cols = ['Pclass', 'Embarked', 'Title']
train_data = pd.get_dummies(train_data, columns=categorical_cols)
test_data = pd.get_dummies(test_data, columns=categorical_cols)
test_data = test_data.reindex(columns=train_data.drop('Survived', axis=1).columns, fill_value=0)

scale_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'TicketGroupSize']

scaler = StandardScaler()
train_data[scale_cols] = scaler.fit_transform(train_data[scale_cols])
test_data[scale_cols] = scaler.transform(test_data[scale_cols])

X_train = train_data.drop(['Survived'], axis=1).astype('float32')
y_train = train_data['Survived'].astype('float32')
X_test = test_data.astype('float32')

X_train.shape, X_test.shape

((891, 18), (418, 18))

Holdout-сплит для быстрой итерации по архитектуре, переведём данные в тензоры и соберём DataLoader для трейна (батчи и шаффл), вал держим одним тензором, тк для оценки батчи не нужны

In [3]:
test_size = 0.2
X_train_holdout, X_val_holdout, y_train_holdout, y_val_holdout = train_test_split(
    X_train, y_train, test_size=test_size, random_state=42, stratify=y_train
)

X_train_t = torch.tensor(X_train_holdout.values)
y_train_t = torch.tensor(y_train_holdout.values).unsqueeze(1)
X_val_t = torch.tensor(X_val_holdout.values)
y_val_t = torch.tensor(y_val_holdout.values).unsqueeze(1)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

X_train_t.shape, X_val_t.shape, y_train_t.shape, y_val_t.shape

(torch.Size([712, 18]),
 torch.Size([179, 18]),
 torch.Size([712, 1]),
 torch.Size([179, 1]))

Соберём функцию обучения и оценки, переиспользуем её для всех дальнейших экспериментов (архитектура, batchnorm, dropout, оптимизаторы и тд), чтобы не копировать цикл обучения каждый раз

In [4]:
def train_and_evaluate_nn(model, train_loader, X_val, y_val, optimizer, n_epochs=50, scheduler=None, loss_fn=None):
    model = model.to(device)
    X_val, y_val = X_val.to(device), y_val.to(device)
    loss_fn = loss_fn or nn.BCEWithLogitsLoss()

    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        if scheduler is not None:
            scheduler.step()

    model.eval()
    with torch.no_grad():
        val_pred = (torch.sigmoid(model(X_val)) >= 0.5).float()
        val_acc = (val_pred == y_val).float().mean().item()

    return model, val_acc

Пункт 1: простая MLP из 2 линейных слоёв с функцией активации между ними. Возвращаем логиты (без сигмоиды на выходе), тк BCEWithLogitsLoss сам применяет sigmoid внутри и делает это численно стабильнее

In [5]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=16):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        return self.fc2(x)


torch.manual_seed(42)
model = SimpleMLP(input_dim=X_train_t.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model, val_acc = train_and_evaluate_nn(model, train_loader, X_val_t, y_val_t, optimizer)
print(f"Скор на holdout-валидации: {val_acc:.5f}")

Скор на holdout-валидации: 0.83799


Простая MLP (16 нейронов, ReLU, Adam lr=1e-3, 50 эпох) дала 0.838 на holdout. Уже сравнимо с логрегом (0.834) и близко к Random Forest (0.839), хотя скор не совсем сопоставим напрямую: это один holdout-сплит, а не 5-fold CV, как у остальных моделей

Пункт 2: добавим больше слоёв. Ширину слоёв фиксируем на 16 (как в базовой MLP из пункта 1), меняем только глубину, чтобы не путать эффект от количества слоёв с эффектом от их размера, размер слоёв отдельно проверим в пункте 5. Обобщим SimpleMLP до произвольного числа скрытых слоёв через hidden_dims. Заведём dnn_results для сравнения всех DNN-экспериментов в конце

In [6]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(16,), activation=nn.ReLU):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(activation())
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


dnn_results = {'mlp_16': val_acc}  # результат SimpleMLP из пункта 1

In [7]:
architectures = {
    'mlp_16_16': (16, 16),
    'mlp_16_16_16': (16, 16, 16),
    'mlp_16_16_16_16': (16, 16, 16, 16),
}

for name, hidden_dims in architectures.items():
    torch.manual_seed(42)
    model = MLP(input_dim=X_train_t.shape[1], hidden_dims=hidden_dims)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    _, val_acc = train_and_evaluate_nn(model, train_loader, X_val_t, y_val_t, optimizer)
    dnn_results[name] = val_acc
    print(f"{name}: {val_acc:.5f}")

mlp_16_16: 0.81006
mlp_16_16_16: 0.79888
mlp_16_16_16_16: 0.79888


При фиксированной ширине (16) рост глубины монотонно ухудшает скор: 0.810 (2 слоя) -> 0.799 (3 слоя) -> 0.799 (4 слоя), и всё хуже базовой 1-слойной MLP (0.838). Значит дело именно в глубине, а не в увеличении числа параметров вместе с шириной, как было в первой версии эксперимента. На 712 строках трейна без регуляризации дополнительные слои только мешают обучению

Пункт 3: отдельный класс MLPWithBatchNorm (Linear -> BatchNorm -> активация в каждом скрытом слое), не трогаем MLP из пункта 2. Прогоним те же 3 архитектуры, что провалились в пункте 2

In [8]:
class MLPWithBatchNorm(nn.Module):
    def __init__(self, input_dim, hidden_dims=(16,), activation=nn.ReLU):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(activation())
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


for name, hidden_dims in architectures.items():
    torch.manual_seed(42)
    model = MLPWithBatchNorm(input_dim=X_train_t.shape[1], hidden_dims=hidden_dims)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    _, val_acc = train_and_evaluate_nn(model, train_loader, X_val_t, y_val_t, optimizer)
    dnn_results[f'{name}_bn'] = val_acc
    print(f"{name}_bn: {val_acc:.5f}")

mlp_16_16_bn: 0.79330
mlp_16_16_16_bn: 0.77095
mlp_16_16_16_16_bn: 0.78212


BatchNorm снова не помог: 0.793/0.771/0.782 против 0.810/0.799/0.799 без него при той же фиксированной ширине, всё ещё хуже базовой MLP (0.838). Дело не в масштабе активаций (для этого и нужен BatchNorm), а в том, что на 712 строках и батче 32 глубина сама по себе избыточна для этой задачи

Пункт 4: отдельный класс MLPWithDropout (Linear -> активация -> Dropout в каждом скрытом слое), пробуем разные значения dropout на тех же 3 архитектурах

In [9]:
class MLPWithDropout(nn.Module):
    def __init__(self, input_dim, hidden_dims=(16,), activation=nn.ReLU, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


dropout_rates = [0.2, 0.4]

for name, hidden_dims in architectures.items():
    for p in dropout_rates:
        torch.manual_seed(42)
        model = MLPWithDropout(input_dim=X_train_t.shape[1], hidden_dims=hidden_dims, dropout=p)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        _, val_acc = train_and_evaluate_nn(model, train_loader, X_val_t, y_val_t, optimizer)
        key = f'{name}_dropout{p}'
        dnn_results[key] = val_acc
        print(f"{key}: {val_acc:.5f}")

mlp_16_16_dropout0.2: 0.81564
mlp_16_16_dropout0.4: 0.82682
mlp_16_16_16_dropout0.2: 0.82123
mlp_16_16_16_dropout0.4: 0.81006
mlp_16_16_16_16_dropout0.2: 0.81564
mlp_16_16_16_16_dropout0.4: 0.81564


Dropout снова лучше BatchNorm: лучший результат 0.827 на mlp_16_16 с dropout=0.4 (против 0.810 без регуляризации и 0.793 с BatchNorm для той же архитектуры). Для более глубоких сетей (3-4 слоя) dropout поднимает скор до 0.816-0.821, тоже лучше своих не-dropout версий, но всё ещё ниже базовой 1-слойной MLP (0.838). Вывод по глубине: для этой задачи и размера датасета 1 скрытый слой лучше, чем 2+

Пункт 5а: размер единственного скрытого слоя (глубину фиксируем на 1, раз пункты 2-4 показали, что больше слоёв только хуже). mlp_16 с ReLU уже есть из пункта 1

In [10]:
hidden_dim_options = [8, 32, 64]

for hidden_dim in hidden_dim_options:
    torch.manual_seed(42)
    model = MLP(input_dim=X_train_t.shape[1], hidden_dims=(hidden_dim,))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    _, val_acc = train_and_evaluate_nn(model, train_loader, X_val_t, y_val_t, optimizer)
    key = f'mlp_{hidden_dim}'
    dnn_results[key] = val_acc
    print(f"{key}: {val_acc:.5f}")

mlp_8: 0.83799
mlp_32: 0.82123
mlp_64: 0.81006


mlp_8 (0.838) сравнялся с mlp_16, а mlp_32 (0.821) и mlp_64 (0.810) хуже. Тот же паттерн, что и с глубиной: чем больше параметров, тем хуже на 712 строках трейна. 8-16 нейронов в единственном скрытом слое - предел, дальше начинается переобучение

Пункт 5б: разные функции активации при ширине 16 (лучшая из пункта 5а). ReLU уже есть (mlp_16 из пункта 1)

In [11]:
activations = {
    'tanh': nn.Tanh,
    'leaky_relu': nn.LeakyReLU,
    'gelu': nn.GELU,
}

for name, activation in activations.items():
    torch.manual_seed(42)
    model = MLP(input_dim=X_train_t.shape[1], hidden_dims=(16,), activation=activation)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    _, val_acc = train_and_evaluate_nn(model, train_loader, X_val_t, y_val_t, optimizer)
    key = f'mlp_16_{name}'
    dnn_results[key] = val_acc
    print(f"{key}: {val_acc:.5f}")

mlp_16_tanh: 0.84358
mlp_16_leaky_relu: 0.82682
mlp_16_gelu: 0.83240


Tanh дал 0.844 - первый результат среди всех DNN-вариантов, обошедший базовую MLP с ReLU (0.838). GELU почти на уровне ReLU (0.832), LeakyReLU хуже (0.827). Новый лидер по DNN: mlp_16 + Tanh

Пункт 6: разные оптимизаторы на текущем лидере (mlp_16 + Tanh). Adam с lr=1e-3 уже есть (mlp_16_tanh из пункта 5б)

In [12]:
optimizer_ctors = {
    'sgd': lambda params: torch.optim.SGD(params, lr=1e-3),
    'sgd_momentum': lambda params: torch.optim.SGD(params, lr=1e-3, momentum=0.9),
    'rmsprop': lambda params: torch.optim.RMSprop(params, lr=1e-3),
    'adamw': lambda params: torch.optim.AdamW(params, lr=1e-3),
}

for name, optimizer_ctor in optimizer_ctors.items():
    torch.manual_seed(42)
    model = MLP(input_dim=X_train_t.shape[1], hidden_dims=(16,), activation=nn.Tanh)
    optimizer = optimizer_ctor(model.parameters())
    _, val_acc = train_and_evaluate_nn(model, train_loader, X_val_t, y_val_t, optimizer)
    key = f'mlp_16_tanh_{name}'
    dnn_results[key] = val_acc
    print(f"{key}: {val_acc:.5f}")

mlp_16_tanh_sgd: 0.59218
mlp_16_tanh_sgd_momentum: 0.80447
mlp_16_tanh_rmsprop: 0.83799
mlp_16_tanh_adamw: 0.83799


Ни один оптимизатор не обошёл Adam (0.844): чистый SGD провалился (0.592, практически не сдвинулся с места за 50 эпох на lr=1e-3), SGD с momentum лучше, но всё ещё слабо (0.804), RMSprop и AdamW сравнялись с ReLU-базлайном (0.838), но не с Tanh-версией. Adam остаётся лидером, скорее всего потому что использованный lr=1e-3 подобран как раз под него, а не под SGD-семейство, которому обычно нужен на порядок больший lr

Пункт 7: косинусовый scheduler (CosineAnnealingLR) поверх Adam на текущем лидере (mlp_16 + Tanh)

In [13]:
n_epochs = 50

torch.manual_seed(42)
model = MLP(input_dim=X_train_t.shape[1], hidden_dims=(16,), activation=nn.Tanh)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

_, val_acc = train_and_evaluate_nn(model, train_loader, X_val_t, y_val_t, optimizer, n_epochs=n_epochs, scheduler=scheduler)
dnn_results['mlp_16_tanh_cosine'] = val_acc
print(f"mlp_16_tanh_cosine: {val_acc:.5f}")

mlp_16_tanh_cosine: 0.81564


Scheduler ухудшил результат: 0.816 против 0.844 без него. CosineAnnealingLR плавно гасит lr до нуля к последней эпохе, а модель и без scheduler'а нормально сходится за 50 эпох на постоянном lr=1e-3 - шедулер только не даёт доучиться на поздних эпохах. Постоянный lr пока лучше

Пункт 8а: learning rate на текущем лидере (mlp_16 + Tanh + Adam). lr=1e-3 уже есть (0.844)

In [14]:
lr_options = [1e-4, 5e-3]

for lr in lr_options:
    torch.manual_seed(42)
    model = MLP(input_dim=X_train_t.shape[1], hidden_dims=(16,), activation=nn.Tanh)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    _, val_acc = train_and_evaluate_nn(model, train_loader, X_val_t, y_val_t, optimizer)
    key = f'mlp_16_tanh_lr{lr}'
    dnn_results[key] = val_acc
    print(f"{key}: {val_acc:.5f}")

mlp_16_tanh_lr0.0001: 0.74860
mlp_16_tanh_lr0.005: 0.80447


1e-3 остаётся лучшим: lr=1e-4 слишком медленный, не успевает обучиться за 50 эпох (0.749), lr=5e-3 слишком агрессивный, обучение нестабильно (0.804)

Пункт 8б: batch size. batch_size=32 уже есть (0.844), для других размеров нужен свой DataLoader

In [15]:
batch_size_options = [16, 64]

for batch_size in batch_size_options:
    loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    torch.manual_seed(42)
    model = MLP(input_dim=X_train_t.shape[1], hidden_dims=(16,), activation=nn.Tanh)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    _, val_acc = train_and_evaluate_nn(model, loader, X_val_t, y_val_t, optimizer)
    key = f'mlp_16_tanh_batch{batch_size}'
    dnn_results[key] = val_acc
    print(f"{key}: {val_acc:.5f}")

mlp_16_tanh_batch16: 0.83240
mlp_16_tanh_batch64: 0.82123


batch_size=32 остаётся лучшим: 16 (0.832) и 64 (0.821) немного хуже

Пункт 8в: количество эпох. n_epochs=50 уже есть (0.844)

In [16]:
epoch_options = [30, 100]

for n_epochs_opt in epoch_options:
    torch.manual_seed(42)
    model = MLP(input_dim=X_train_t.shape[1], hidden_dims=(16,), activation=nn.Tanh)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    _, val_acc = train_and_evaluate_nn(model, train_loader, X_val_t, y_val_t, optimizer, n_epochs=n_epochs_opt)
    key = f'mlp_16_tanh_epochs{n_epochs_opt}'
    dnn_results[key] = val_acc
    print(f"{key}: {val_acc:.5f}")

mlp_16_tanh_epochs30: 0.81564
mlp_16_tanh_epochs100: 0.82682


50 эпох остаются лучшим: 30 недообучена (0.816), 100 уже немного переобучена (0.827)

Пункт 8г: loss_fn. Классы не сбалансированы (62/38), попробуем BCEWithLogitsLoss с pos_weight (аналог class_weight='balanced' в sklearn) вместо обычного BCEWithLogitsLoss

In [17]:
n_negative = (y_train_holdout == 0).sum()
n_positive = (y_train_holdout == 1).sum()
pos_weight = torch.tensor([n_negative / n_positive], dtype=torch.float32).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

torch.manual_seed(42)
model = MLP(input_dim=X_train_t.shape[1], hidden_dims=(16,), activation=nn.Tanh)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
_, val_acc = train_and_evaluate_nn(model, train_loader, X_val_t, y_val_t, optimizer, loss_fn=loss_fn)
dnn_results['mlp_16_tanh_posweight'] = val_acc
print(f"mlp_16_tanh_posweight: {val_acc:.5f}")

mlp_16_tanh_posweight: 0.83240


pos_weight не помог (0.832 против 0.844) - как и с class_weight='balanced' в классических моделях, дисбаланс 62/38 недостаточно сильный, чтобы взвешивание давало выигрыш. Обычный BCEWithLogitsLoss без веса остаётся лучшим

Пункт 9 (со звёздочкой): Embedding слой для категориальных фичей вместо one-hot. Нужен отдельный препроцессинг (категории как индексы, а не dummy-колонки) и отдельная модель с двумя входами (категориальный + числовой), поэтому переиспользовать X_train/MLP не получится напрямую

In [18]:
train_data_emb = pd.read_csv("data/train.csv")
test_data_emb = pd.read_csv("data/test.csv")

train_data_emb, artifacts_emb = preprocess_data_advanced(train_data_emb, is_train=True)
test_data_emb = preprocess_data_advanced(test_data_emb, is_train=False, artifacts=artifacts_emb)

cat_cols = ['Pclass', 'Embarked', 'Title']
num_cols = ['Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'TicketGroupSize', 'HasCabin']

# категории фиксируем по train, чтобы train и test кодировались одинаковыми индексами
cat_categories = {col: train_data_emb[col].astype('category').cat.categories for col in cat_cols}

for col in cat_cols:
    train_data_emb[col] = pd.Categorical(train_data_emb[col], categories=cat_categories[col]).codes
    test_data_emb[col] = pd.Categorical(test_data_emb[col], categories=cat_categories[col]).codes

scaler_emb = StandardScaler()
train_data_emb[num_cols] = scaler_emb.fit_transform(train_data_emb[num_cols])
test_data_emb[num_cols] = scaler_emb.transform(test_data_emb[num_cols])

train_data_emb.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,TicketGroupSize,HasCabin
0,0,2,-0.737695,-0.557460,0.432793,-0.473674,-0.502445,2,2,-0.579162,-0.544925
1,1,0,1.355574,0.649091,0.432793,-0.473674,0.786845,0,3,-0.579162,1.835115
2,1,2,1.355574,-0.255822,-0.474545,-0.473674,-0.488854,2,1,-0.579162,-0.544925
3,1,0,1.355574,0.422862,0.432793,-0.473674,0.420730,2,3,0.155928,1.835115
4,0,2,-0.737695,0.422862,-0.474545,-0.473674,-0.486337,2,2,-0.579162,-0.544925


In [19]:
X_train_emb_holdout, X_val_emb_holdout, y_train_emb_holdout, y_val_emb_holdout = train_test_split(
    train_data_emb.drop('Survived', axis=1), train_data_emb['Survived'],
    test_size=test_size, random_state=42, stratify=train_data_emb['Survived']
)

x_cat_train = torch.tensor(X_train_emb_holdout[cat_cols].values, dtype=torch.long)
x_num_train = torch.tensor(X_train_emb_holdout[num_cols].values, dtype=torch.float32)
y_train_emb = torch.tensor(y_train_emb_holdout.values, dtype=torch.float32).unsqueeze(1)

x_cat_val = torch.tensor(X_val_emb_holdout[cat_cols].values, dtype=torch.long)
x_num_val = torch.tensor(X_val_emb_holdout[num_cols].values, dtype=torch.float32)
y_val_emb = torch.tensor(y_val_emb_holdout.values, dtype=torch.float32).unsqueeze(1)

x_cat_train.shape, x_num_train.shape

(torch.Size([712, 3]), torch.Size([712, 7]))

In [20]:
class EmbeddingMLP(nn.Module):
    def __init__(self, cat_cardinalities, embedding_dims, num_numeric, hidden_dim=16, activation=nn.Tanh):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, dim) for cardinality, dim in zip(cat_cardinalities, embedding_dims)
        ])
        total_embedding_dim = sum(embedding_dims)
        self.fc1 = nn.Linear(total_embedding_dim + num_numeric, hidden_dim)
        self.activation = activation()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x_cat, x_num):
        embedded = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embedded + [x_num], dim=1)
        x = self.fc1(x)
        x = self.activation(x)
        return self.fc2(x)


def train_and_evaluate_embedding_model(model, x_cat_train, x_num_train, y_train, x_cat_val, x_num_val, y_val,
                                        optimizer, n_epochs=50, batch_size=32, loss_fn=None):
    model = model.to(device)
    loss_fn = loss_fn or nn.BCEWithLogitsLoss()
    loader = DataLoader(TensorDataset(x_cat_train, x_num_train, y_train), batch_size=batch_size, shuffle=True)
    x_cat_val, x_num_val, y_val = x_cat_val.to(device), x_num_val.to(device), y_val.to(device)

    for epoch in range(n_epochs):
        model.train()
        for xc, xn, yb in loader:
            xc, xn, yb = xc.to(device), xn.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xc, xn), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        val_pred = (torch.sigmoid(model(x_cat_val, x_num_val)) >= 0.5).float()
        val_acc = (val_pred == y_val).float().mean().item()

    return model, val_acc

In [21]:
cat_cardinalities = [len(cat_categories[col]) for col in cat_cols]
embedding_dims = [max(2, cardinality // 2) for cardinality in cat_cardinalities]

torch.manual_seed(42)
model = EmbeddingMLP(cat_cardinalities, embedding_dims, num_numeric=len(num_cols), hidden_dim=16, activation=nn.Tanh)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model, val_acc = train_and_evaluate_embedding_model(
    model, x_cat_train, x_num_train, y_train_emb, x_cat_val, x_num_val, y_val_emb, optimizer
)
dnn_results['mlp_16_tanh_embedding'] = val_acc
print(f"mlp_16_tanh_embedding: {val_acc:.5f}")

mlp_16_tanh_embedding: 0.84916


Embedding дал 0.849 - новый лидер среди всех DNN-вариантов, обошёл лучший one-hot результат (0.844). Embedding превращает Pclass/Embarked/Title в плотные векторы (2 числа на категорию вместо 3-5 бинарных колонок), сеть сама учится, какие категории похожи друг на друга, а не считает их независимыми осями, как one-hot

Докрутим EmbeddingMLP: переберём размер эмбеддингов (одинаковый для всех 3 категориальных колонок) и hidden_dim вокруг текущих значений (2 и 16)

In [22]:
embedding_dim_options = [2, 3, 4]
hidden_dim_options = [8, 16, 32]

for emb_dim in embedding_dim_options:
    for hidden_dim in hidden_dim_options:
        torch.manual_seed(42)
        embedding_dims = [emb_dim] * len(cat_cardinalities)
        model = EmbeddingMLP(cat_cardinalities, embedding_dims, num_numeric=len(num_cols),
                              hidden_dim=hidden_dim, activation=nn.Tanh)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        _, val_acc = train_and_evaluate_embedding_model(
            model, x_cat_train, x_num_train, y_train_emb, x_cat_val, x_num_val, y_val_emb, optimizer
        )
        key = f'embedding{emb_dim}_hidden{hidden_dim}'
        dnn_results[key] = val_acc
        print(f"{key}: {val_acc:.5f}")

embedding2_hidden8: 0.82123
embedding2_hidden16: 0.84916
embedding2_hidden32: 0.83240
embedding3_hidden8: 0.82123
embedding3_hidden16: 0.82123
embedding3_hidden32: 0.82682
embedding4_hidden8: 0.81564
embedding4_hidden16: 0.83799
embedding4_hidden32: 0.83240


Ни одна из 9 комбинаций не обошла исходную (embedding_dim=2, hidden=16, 0.849) - она же оказалась лучшей и в этом переборе. И увеличение эмбеддингов (3-4), и увеличение/уменьшение hidden_dim только ухудшают результат. Похоже, это локальный максимум для данного объёма данных: 2 числа на категорию уже достаточно, чтобы закодировать Pclass/Embarked/Title (3, 3 и 5 значений), а больше параметров снова ведёт к переобучению, как и в пунктах 2-5. Финальная конфигурация DNN: EmbeddingMLP(embedding_dim=2, hidden_dim=16, Tanh, Adam lr=1e-3, batch=32, 50 эпох) = 0.849